In [ ]:
#2.2编写一个函数 preprocess_text(text, n)，完成以下步骤：
import re
from collections import Counter
def preprocess_text(text, n):
    #1.将文本转换为小写，去除标点符号（保留字母和空格）。
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # 只保留小写字母和空格
    
    #2.按空格分词。
    words = text.split()  # 自动处理多个空格
    
    #3.构建词汇表（按出现频率排序，分配整数 ID，从 0 开始）。
    # 构建词汇表：按频率排序，分配ID
    word_counts = Counter(words)
    # 按频率降序，频率相同按字母序（可选）
    sorted_vocab = sorted(word_counts.items(), key=lambda x: (-x[1], x[0]))
    vocab = {word: idx for idx, (word, _) in enumerate(sorted_vocab)}
    
    #4.用滑动窗口生成长度为 n 的特征序列和对应的下一个词标签（用于自回归语言模型）
    features = []
    labels = []
    if len(words) <= n:
        # 如果词数不足 n+1，没有足够样本
        return vocab, ([], [])
    for i in range(len(words) - n + 1):  # 修改这里，+1 以包含最后一个窗口
        feature = words[i:i+n]
        # 判断是否有下一个词
        if i + n < len(words):
            label = words[i+n]
        else:
            label = None
        features.append(feature)
        labels.append(label)
    return vocab, (features, labels)

# 测试
text = "The time machine"
n = 2
vocab, (features, labels) = preprocess_text(text, n)
print("词汇表:", vocab)
print("特征:", features)
print("标签:", labels)

词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征: [['the', 'time'], ['time', 'machine']]
标签: ['machine', None]


In [5]:
#3.2实现一个简单的 RNN 单元的前向传播和单步反向传播（仅计算梯度，不更新）。给定输入 x_t（形状 (batch_size, input_size)）、上一隐藏状态 h_prev（形状 (batch_size, hidden_size)），以及权重 W_hx, W_hh,b_h，计算当前隐藏状态 h_t。同时实现反向传播，已知上游梯度 dh_next（即损失对 h_t 的梯度），计算 dx_t, dh_prev, dW_hx, dW_hh, db_h（使用tanh 激活函数）。
import numpy as np

def rnn_step_forward(x, h_prev, W_hx, W_hh, b_h):
    """
    前向传播：h_t = tanh(W_hx x + W_hh h_prev + b_h)
    x: (batch, input_size)
    h_prev: (batch, hidden_size)
    W_hx: (hidden, input)
    W_hh: (hidden, hidden)
    b_h: (hidden,)
    返回 h_next, cache (用于反向传播)
    """
    a = np.dot(x, W_hx.T) + np.dot(h_prev, W_hh.T) + b_h  # (batch, hidden)
    h_next = np.tanh(a)
    cache = (x, h_prev, W_hx, W_hh, b_h, a, h_next)
    return h_next, cache

def rnn_step_backward(dh_next, cache):
    """
    反向传播：已知 dh_next = dL/dh_next (形状 batch, hidden)
    返回 dx, dh_prev, dW_hx, dW_hh, db_h
    """
    x, h_prev, W_hx, W_hh, b_h, a, h_next = cache
    # tanh 导数: dtanh = 1 - tanh^2
    da = dh_next * (1 - h_next ** 2)  # (batch, hidden)
    
    # 对各参数求导
    db_h = np.sum(da, axis=0)  # (hidden,)
    dW_hh = np.dot(da.T, h_prev)  # (hidden, hidden)
    dW_hx = np.dot(da.T, x)       # (hidden, input)
    dh_prev = np.dot(da, W_hh)    # (batch, hidden)
    dx = np.dot(da, W_hx)         # (batch, input)
    
    return dx, dh_prev, dW_hx, dW_hh, db_h

# 测试
batch_size, input_size, hidden_size = 2, 3, 4
x = np.random.randn(batch_size, input_size)
h_prev = np.random.randn(batch_size, hidden_size)
W_hx = np.random.randn(hidden_size, input_size)
W_hh = np.random.randn(hidden_size, hidden_size)
b_h = np.random.randn(hidden_size)

h_next, cache = rnn_step_forward(x, h_prev, W_hx, W_hh, b_h)
dh_next = np.random.randn(batch_size, hidden_size)
dx, dh_prev, dW_hx, dW_hh, db_h = rnn_step_backward(dh_next, cache)

print("前向输出 h_next shape:", h_next.shape)
print("dx shape:", dx.shape)
print("dh_prev shape:", dh_prev.shape)
print("dW_hx shape:", dW_hx.shape)
print("dW_hh shape:", dW_hh.shape)
print("db_h shape:", db_h.shape)


前向输出 h_next shape: (2, 4)
dx shape: (2, 3)
dh_prev shape: (2, 4)
dW_hx shape: (4, 3)
dW_hh shape: (4, 4)
db_h shape: (4,)


In [ ]:
#4.2实现一个双向 RNN 编码器，接收序列 X（形状 (seq_len, batch,input_dim)），使用 torch.nn.RNN 或手动实现。要求返回每个时间步的拼接后的前向和后向隐藏状态（形状 (seq_len, batch, 2*hidden_dim)），以及最终时间步的拼接隐藏状态（作为序列表示）。
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=False,  # 输入形状 (seq, batch, feature)
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.hidden_dim = hidden_dim
        
    def forward(self, X):
        # X: (seq_len, batch, input_dim)
        outputs, h_n = self.rnn(X)  # outputs: (seq_len, batch, num_directions * hidden_dim)
        # 每个时间步的拼接隐藏状态就是 outputs
        # 最终时间步的拼接隐藏状态：取 outputs[-1] 或者用 h_n
        # h_n 形状 (num_layers * num_directions, batch, hidden_dim)
        # 取最后一层两个方向的拼接
        # 方法1：从 outputs 取最后一个时间步
        final_state = outputs[-1]  # (batch, 2*hidden_dim)
        return outputs, final_state

# 测试
seq_len, batch, input_dim = 5, 3, 10
hidden_dim = 4
X = torch.randn(seq_len, batch, input_dim)
encoder = BiRNNEncoder(input_dim, hidden_dim)
outputs, final = encoder(X)
print("outputs shape:", outputs.shape)  
print("final shape:", final.shape)

outputs shape: torch.Size([5, 3, 8])
final shape: torch.Size([3, 8])


In [7]:
#5.2实现 CBOW 模型的前向传播和损失计算（不使用负采样，仅用完整softmax）。给定一批上下文词的索引列表（每个样本有 context_size 个上下文词），词汇表大小 V，嵌入维度 d。输入权重矩阵 W（形状 (V, d)）和输出权重矩阵 W_out（形状 (d, V)）。计算平均上下文向量作为隐藏层，然后计算输出概率分布，并计算交叉熵损失（目标为中心词索引）。返回损失值。
import numpy as np
def cbow_forward(context_indices, target_indices, W, W_out):
    """
    context_indices: (batch_size, context_size) 每个样本的上下文词索引
    target_indices: (batch_size,) 每个样本的中心词索引
    W: (V, d) 输入嵌入矩阵
    W_out: (d, V) 输出权重矩阵
    返回损失值（标量）
    """
    batch_size, context_size = context_indices.shape
    V, d = W.shape
    
    # 获取上下文词向量 (batch, context_size, d)
    # 使用 np.take 或 advanced indexing
    embeds = W[context_indices]  # (batch, context_size, d)
    # 平均得到隐藏层 h (batch, d)
    h = np.mean(embeds, axis=1)  # (batch, d)
    
    # 计算分数 logits (batch, V)
    logits = np.dot(h, W_out)  # (batch, V)
    
    # softmax 得到概率
    exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))  # 数值稳定
    probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)  # (batch, V)
    
    # 交叉熵损失：对每个样本，取目标索引的概率的负对数，然后平均
    # 提取目标词的概率
    batch_indices = np.arange(batch_size)
    target_probs = probs[batch_indices, target_indices]  # (batch,)
    loss = -np.mean(np.log(target_probs + 1e-10))  # 加小值防止 log(0)
    
    return loss

# 测试
V, d = 10, 4
batch_size, context_size = 3, 2
W = np.random.randn(V, d)
W_out = np.random.randn(d, V)
context_indices = np.random.randint(0, V, (batch_size, context_size))
target_indices = np.random.randint(0, V, batch_size)

loss = cbow_forward(context_indices, target_indices, W, W_out)
print("CBOW loss:", loss)

CBOW loss: 2.8157125065233295


In [9]:
#6.2实现多头注意力（Multi-Head Attention）的前向传播，假设 num_heads=2，d_model=4。给定输入 X（形状 (seq_len, batch, d_model)），分别线性投影得到 Q, K, V（每个头的维度 d_k = d_v = d_model/num_heads）。对每个头计算缩放点积注意力，然后将所有头的输出拼接并经过最终线性层。返回输出（形状与输入相同）。
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.d_v = d_model // num_heads
        
        # 线性投影层
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        
    def forward(self, X):
        # X: (seq_len, batch, d_model)
        seq_len, batch, _ = X.shape
        
        # 线性投影并分割成多头
        Q = self.W_q(X)  # (seq_len, batch, d_model)
        K = self.W_k(X)
        V = self.W_v(X)
        
        # 重塑为 (seq_len, batch, num_heads, d_k) 然后转置为 (batch, num_heads, seq_len, d_k)
        Q = Q.view(seq_len, batch, self.num_heads, self.d_k).transpose(0, 1).transpose(1, 2)  # (batch, num_heads, seq_len, d_k)
        K = K.view(seq_len, batch, self.num_heads, self.d_k).transpose(0, 1).transpose(1, 2)
        V = V.view(seq_len, batch, self.num_heads, self.d_v).transpose(0, 1).transpose(1, 2)
        
        # 缩放点积注意力
        # scores: (batch, num_heads, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        attn_weights = F.softmax(scores, dim=-1)  # 对最后一个维度 (键) 做 softmax
        # 加权求和
        attn_output = torch.matmul(attn_weights, V)  # (batch, num_heads, seq_len, d_v)
        
        # 合并多头: 转置回 (seq_len, batch, num_heads, d_v) 并合并
        attn_output = attn_output.transpose(1, 2).transpose(0, 1).contiguous()  # (seq_len, batch, num_heads, d_v)
        attn_output = attn_output.view(seq_len, batch, self.d_model)  # (seq_len, batch, d_model)
        
        # 最终线性层
        output = self.W_o(attn_output)  # (seq_len, batch, d_model)
        return output

# 测试
d_model = 4
num_heads = 2
seq_len, batch = 3, 2
X = torch.randn(seq_len, batch, d_model)
mha = MultiHeadAttention(d_model, num_heads)
out = mha(X)
print("输入形状:", X.shape)
print("输出形状:", out.shape)  # 应为 (3, 2, 4)

输入形状: torch.Size([3, 2, 4])
输出形状: torch.Size([3, 2, 4])
